# Import

In [29]:
import math
import torch
import torch.nn as nn

Input

|

MultiheadAttention

|

Add(Residual) (just adding x + attention_output) 

|

LayerNorm (just a normalization layer)

|

Feed Forward Network

|

Add (Residual)

|

LayerNorm

# What is FFN Acutally do?

#### FFN does:

- Per Token Computation

#### Think:

- Attention = Communication

- FFN = Thinking

In [30]:
# Example

# After attention:

# I    -> [0.1, 0.2, ...]
# love -> [0.5, 0.8, ...]
# AI   -> [0.7, 0.3, ...]

# FFN processes each token independently:

# I     gets processed
# love  gets processed
# AI    gets processed

# No token interaction here.

#_____________________________________________________

# FFN Architecture

# In the Transformer paper:

# Linear
#  ↓
# ReLU
#  ↓
# Linear

#______________________________________________________

# Modern Transformers usually use:

# Linear
#  ↓
# GELU
#  ↓
# Linear

In [31]:
class MultiHeadAttention(nn.Module):
    
    def __init__(self, d_model, num_heads):
        super().__init__()
        
        # assert is keyword
        # its a debugging tools used to test if a specific condition in our code evalutes to true
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads


        self.Wq = nn.Linear(in_features = d_model, out_features = d_model)
        self.Wk = nn.Linear(in_features = d_model, out_features = d_model)
        self.Wv = nn.Linear(in_features = d_model, out_features = d_model)

        self.out_proj = nn.Linear(in_features = d_model, out_features = d_model)

    def forward(self, x):

        batch_size, seq_len, d_model = x.shape

        print('=' * 10)
        print("x shape ", x.shape)

        #send x in linear layer

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        print("\n After Linear Layers")

        print("Q : ", Q.shape)
        print("K : ", K.shape)
        print("V : ", V.shape)

        # split into heads using view
        # view is like reshaping but without copy 

        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )
        

        print("\n After reshaping")
        print(Q.shape)
        print(K.shape)
        print(V.shape)

        # Move heads forward
        
        # change shape 
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        print("\n After Transpose")
        print("Q : ", Q.shape)
        print("K : ", K.shape)
        print("V : ", V.shape)

        # Attention score

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dim)

        print("\nscore")
        print(scores.shape)

        # softmax

        weights = torch.softmax(scores, dim = -1)

        print("\nAfter softmax Weight Shape")
        print(weights.shape)

        print("\nHead 1 Attentation matrix Weights")
        print(weights[0, 0])
        print("\nHead 2 Attentation matrix Weights")
        print(weights[0, 1])
        
        # Apply Attention
        output = weights @ V

        print("\nAfter Weights @ V")
        print(output.shape)

        # Combine heads
        output = output.transpose(1, 2)

        print("\nAfter Transpose Back")
        print(output.shape)

        # After Transpose we can't use view to reshape it 
        # so contiguous allocates a new block of memory to copy and rearrange a tensor's data into a sequential, unbroken memory layout.
        # then apply view()

        # print(output.contiguous())

        output = output.contiguous().view(
            batch_size,
            seq_len,
            d_model
        )

        # After Flatten
        print("\nAfter Flatten heads (2, 4 to 8)")
        print(output.shape)

        # Final projection

        output = self.out_proj(output)

        print("\nFinal output")
        print(output.shape)

        return output 

# v6 FeedForawd Network

In [32]:
class FeedForwardNetwork(nn.Module):

    def __init__(self, d_model):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(in_features = d_model, out_features = d_model * 4),

            nn.GELU(),

            nn.Linear(in_features = d_model * 4, out_features = d_model)
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
class TransformerEncoderBlock(nn.Module):
    
    def __init__(self, d_model, num_heads):

        super().__init__()

        # created a obj for MHA
        self.mha = MultiHeadAttention(d_model = d_model, num_heads = num_heads)

        # LayerNorm from nn (new thing in V5)
        self.norm1 = nn.LayerNorm(d_model)

        # V6

        # created a obj for FFN
        self.ffn = FeedForwardNetwork(d_model)

        # LayerNorm
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):

        print("\n Input x")
        print(x)

        # giving value for the obj
        attention_output = self.mha(x)

        print("\nAttention output")
        print(attention_output.shape)

        # residual
        x += attention_output # Residual (new thing in V5)

        print("\nresidual output")
        print(x.shape)

        # normalization
        x = self.norm1(x)

        print("\nNormalization (LayerNorm)")
        print(x.shape)

        # FFN Block

        ffn_output = self.ffn(x)

        print("\n After FFN")
        print(ffn_output.shape)

        # Residual
        x += ffn_output

        print("\nAfter Add Residual")
        print(x.shape)

        # Normalization

        x = self.norm2(x)
        
        print("\nafter LayerNorm")
        print(x.shape)


        return x



In [ ]:
# Example input

batch_size = 1
seq_len = 3
d_model = 8
num_heads = 2

vocab = {
    "I" : 0,
    "Love" : 1,
    "AI" : 2
}

tokens = torch.tensor([0, 1, 2]) # or torch.tensor(vocab.values())

embedding = nn.Embedding(
    num_embeddings = len(vocab),
    embedding_dim = d_model
)

x  = embedding(tokens)

x = x.view(1, 3, 8)


block = TransformerEncoderBlock(d_model = d_model, num_heads = num_heads)

output = block(x)

print("\noutput")
print(output.shape)


 Input x
tensor([[[-0.0019, -0.6090,  1.3927, -0.3576, -0.5642, -0.5425, -0.7584,
          -0.1867],
         [-0.9620,  0.8039, -0.7321, -0.3963,  0.1877, -1.4204, -1.1468,
           0.1808],
         [-1.3799,  0.9914, -0.3261, -0.0686,  1.2744,  1.0514,  0.9146,
          -1.3006]]], grad_fn=<ViewBackward0>)
x shape  torch.Size([1, 3, 8])

 After Linear Layers
Q :  torch.Size([1, 3, 8])
K :  torch.Size([1, 3, 8])
V :  torch.Size([1, 3, 8])

 After reshaping
torch.Size([1, 3, 2, 4])
torch.Size([1, 3, 2, 4])
torch.Size([1, 3, 2, 4])

 After Transpose
Q :  torch.Size([1, 2, 3, 4])
K :  torch.Size([1, 2, 3, 4])
V :  torch.Size([1, 2, 3, 4])

score
torch.Size([1, 2, 3, 3])

After softmax Weight Shape
torch.Size([1, 2, 3, 3])

Head 1 Attentation matrix Weights
tensor([[0.2527, 0.3383, 0.4090],
        [0.5766, 0.2286, 0.1948],
        [0.2864, 0.3636, 0.3500]], grad_fn=<SelectBackward0>)

Head 2 Attentation matrix Weights
tensor([[0.3891, 0.3482, 0.2627],
        [0.2622, 0.4415, 0.296

In [35]:
print(x.mean())
print(x.std())


print(output.shape)
print(output.mean())
print(output.std())

tensor(-0.1893, grad_fn=<MeanBackward0>)
tensor(0.9313, grad_fn=<StdBackward0>)
torch.Size([1, 3, 8])
tensor(4.9671e-09, grad_fn=<MeanBackward0>)
tensor(1.0215, grad_fn=<StdBackward0>)
